# STT 05 — EDA du grand dataset (version Google Colab) (*Scripted Speech 27.0* FR, 28,5 Go)

**Pourquoi cette EDA :** toutes nos décisions de données (vocabulaire, durées, contrainte du CTC, locuteurs) reposent sur l'EDA du **petit** dataset (19 h). Avant d'entraîner sur le grand dataset, on **mesure** : le nombre exact d'heures et de clips, et si nos choix restent valables. Chaque point aboutit à une décision.

| Point | Question | Décision qui en dépend |
|---|---|---|
| 0 | Structure, temps de téléchargement, place sur le disque | Faisabilité sur Kaggle |
| 1 | Colonnes, phrases manquantes, lecture des `.tsv` | Ce qu'on garde ; `load_split` à corriger ? |
| 3–4 | Heures exactes, durées | Taille du sous-ensemble, mémoire GPU, durée d'une époque |
| 2 | Caractères | Garder nos 49 caractères ou élargir le vocabulaire |
| 5 | Contrainte du CTC | Garder la réduction ÷ 4 |
| 6 | Locuteurs | Plafonner ou non les locuteurs dominants |
| 7a | Recouvrement dev/test ↔ train, et réserve de `validated` | Combien d'heures utilisables sans triche |
| 7b | Recouvrement avec **nos** données actuelles | Peut-on garder notre ancien dev pour comparer ? |
| 4 | Fréquences d'échantillonnage, vitesse de lecture | Le DataLoader sera-t-il un goulot ? |

**Avant de lancer (Google Colab) :**
1. **Exécution → Modifier le type d'exécution → CPU** : pas besoin de GPU.
2. **La clé :** dans la barre de gauche, clique sur l'icône 🔑 **Secrets**, puis **+ Ajouter un secret**. Nom : `MDC_API_KEY`, valeur : ta clé MDC. Active ensuite **« Accès au notebook »**.
3. **Exécution → Tout exécuter** (≈ 30 à 50 min).
4. **Garde l'onglet ouvert** jusqu'à la fin : sur Colab, il n'y a pas d'exécution en arrière-plan. Si l'onglet est fermé, ou s'il reste inactif trop longtemps, la session est coupée.

**Règles de données (conditions MDC) :** les données vont dans `/tmp`, qui n'est jamais sauvegardé avec le notebook. Aucune phrase n'est affichée, seulement des nombres et des caractères isolés. Le notebook reste **privé** (dans ton Google Drive, non partagé). Ne copie **jamais** les données dans Google Drive.

**À renvoyer :** toute la sortie, ou au minimum celle de la dernière cellule (le fichier `eda_grand_dataset.json` se télécharge automatiquement à la fin).

## Cellule 1 — Notre code

On clone notre branche pour réutiliser **exactement** notre normalisation du texte (`stt/text.py`) et notre lecture audio (`stt/audio.py`). Ainsi, l'EDA mesure ce que l'entraînement verra vraiment.

In [ ]:
!git clone -b stt-transformer-junior --depth 1 https://github.com/mohamed-943/projet-J-ONE.git
%cd projet-J-ONE/recherche/stt_transformer
!pip install -q datacollective soundfile

## Cellule 2 — Point 0 : téléchargement, décompression, structure

**Mesures :**
- le **temps de téléchargement** et de **décompression** : ce coût se répètera à chaque session d'entraînement ;
- la **place sur le disque** avant et après ;
- la **liste des `.tsv`**. Si `clip_durations.tsv` existe, il donnera les heures exactes sans ouvrir chaque mp3 ;
- le **nombre de mp3**.

Si la place manque pour tout décompresser, la cellule n'extrait que les `.tsv`. Tous les points restent alors faisables, sauf la vitesse de lecture (cellule 10).

In [ ]:
import os, glob, time, json, shutil, subprocess
from google.colab import userdata

# La clé est lue dans les Secrets de Colab (icône 🔑) : elle n'apparaît jamais dans le code
os.environ["MDC_API_KEY"] = userdata.get("MDC_API_KEY")
from datacollective import download_dataset

SUMMARY = {}                                   # tous les chiffres de l'EDA (aucune phrase), sauvegardés à la fin
BIG_ID = "cmu5kub4600pkmh078n7n5ogl"           # Scripted Speech 27.0 FR
BIG_DIR = "/tmp/big"                           # /tmp : jamais sauvegardé avec le notebook (conditions MDC)

def free_gb():
    return shutil.disk_usage("/tmp").free / 1e9

print(f"Place libre au départ : {free_gb():.0f} Go")
start = time.time()
download_dataset(BIG_ID, download_directory=BIG_DIR)
archive = glob.glob(f"{BIG_DIR}/*.tar.gz")[0]
SUMMARY["archive_go"] = round(os.path.getsize(archive) / 1e9, 2)
SUMMARY["telechargement_min"] = round((time.time() - start) / 60, 1)
print(f"Téléchargé : {SUMMARY['archive_go']} Go en {SUMMARY['telechargement_min']} min | place libre : {free_gb():.0f} Go")

# Décompression complète s'il reste la place (les mp3 sont déjà compressés : contenu ≈ taille de l'archive),
# sinon seulement les .tsv
start = time.time()
if free_gb() > 1.1 * SUMMARY["archive_go"]:
    subprocess.run(["tar", "-xzf", archive, "-C", BIG_DIR], check=True)
    SUMMARY["extraction"] = "complète"
else:
    subprocess.run(["tar", "-xzf", archive, "-C", BIG_DIR, "--wildcards", "*.tsv"], check=True)
    SUMMARY["extraction"] = "tsv seulement (place insuffisante)"
SUMMARY["extraction_min"] = round((time.time() - start) / 60, 1)
os.remove(archive)                             # libère les 28,5 Go de l'archive
print(f"Extraction {SUMMARY['extraction']} en {SUMMARY['extraction_min']} min | place libre : {free_gb():.0f} Go")

# Structure : les .tsv et leur taille, le nombre de mp3
BASE = os.path.dirname(glob.glob(f"{BIG_DIR}/**/train.tsv", recursive=True)[0])
print("Dossier :", BASE)
for path in sorted(glob.glob(f"{BASE}/*.tsv")):
    print(f"{os.path.getsize(path) / 1e6:9.1f} Mo  {os.path.basename(path)}")
if os.path.isdir(f"{BASE}/clips"):
    SUMMARY["fichiers_mp3"] = len(os.listdir(f"{BASE}/clips"))
    print(f"{SUMMARY['fichiers_mp3']:,} fichiers mp3")

## Cellule 3 — Point 1 : fichiers, colonnes, transcriptions manquantes

On lit chaque tableau (`train`, `dev`, `test`, `validated`…) et on compte les clips et les phrases manquantes.

**Vérification importante :** notre `load_split` lit les `.tsv` avec le réglage par défaut de pandas, qui traite les guillemets comme des délimiteurs. Si une phrase contient un guillemet, des lignes peuvent être **fusionnées sans erreur visible**. On compare donc les deux lectures : si les nombres diffèrent, il faudra corriger `load_split` avant l'entraînement.

In [ ]:
# Cellule 3 : point 1, fichiers, colonnes et transcriptions manquantes
import csv
import pandas as pd

# Les tableaux du dataset qui contiennent des clips, s'ils existent
SPLITS = [s for s in ["train", "dev", "test", "validated", "other", "invalidated"]
          if os.path.exists(f"{BASE}/{s}.tsv")]

def read_tsv(name):
    # QUOTE_NONE : un guillemet dans une phrase est un caractère normal, pas un délimiteur
    return pd.read_csv(f"{BASE}/{name}.tsv", sep="\t", quoting=csv.QUOTE_NONE, low_memory=False)

tables = {s: read_tsv(s) for s in SPLITS}
print("Tableaux trouvés :", SPLITS)
for s, df in tables.items():
    missing = int(df["sentence"].isna().sum())
    print(f"{s:12s} {len(df):>9,} clips | {missing:>6,} phrases manquantes")
    SUMMARY[f"{s}_clips"] = len(df)
    SUMMARY[f"{s}_phrases_manquantes"] = missing

print("\nColonnes :", list(tables["train"].columns))
print("Valeurs manquantes par colonne (train) :")
print(tables["train"].isna().sum().to_string())

# Notre load_split lit les .tsv avec le réglage par défaut de pandas (guillemets actifs).
# Si une phrase contient un guillemet, des lignes peuvent être fusionnées : on compare les deux lectures.
default_rows = len(pd.read_csv(f"{BASE}/train.tsv", sep="\t", low_memory=False))
print(f"\nLignes du train : {default_rows:,} (lecture par défaut) contre {len(tables['train']):,} (guillemets ignorés)")
SUMMARY["train_lignes_lecture_defaut"] = default_rows

# Pour la suite, on retire les clips sans phrase
tables = {s: df.dropna(subset=["sentence"]).reset_index(drop=True) for s, df in tables.items()}

## Cellule 4 — Points 3 et 4 : durées et nombre exact d'heures

Pour chaque tableau : les heures totales, la durée médiane, minimale et maximale, et la durée sous laquelle se trouvent 99 % des clips.

**À regarder :**
- les **heures du `train` officiel** comparées à celles de **`validated`**. Dans Common Voice, le train officiel ne garde souvent qu'un enregistrement par phrase, et il peut être bien plus petit que l'ensemble des clips validés ;
- les clips de **plus de 15 s** : ils coûtent cher en mémoire GPU (l'attention croît avec le carré de la longueur).

L'histogramme est sauvegardé dans `/content/eda_durees.png` (panneau 📁 Fichiers).

In [ ]:
# Cellule 4 : points 3 et 4, durées et nombre d'heures
import numpy as np
import soundfile as sf

dur_path = f"{BASE}/clip_durations.tsv"
if os.path.exists(dur_path):
    d = pd.read_csv(dur_path, sep="\t", quoting=csv.QUOTE_NONE)
    print("Colonnes de clip_durations.tsv :", list(d.columns))
    durations = dict(zip(d.iloc[:, 0], d.iloc[:, 1] / 1000))     # nom du mp3 -> secondes (fichier en millisecondes)
    SUMMARY["source_durees"] = "clip_durations.tsv"
else:
    # Pas de fichier de durées : on lit l'en-tête de chaque mp3 (plus lent)
    print("clip_durations.tsv absent : lecture des en-têtes mp3")
    durations = {}
    for p in set().union(*[set(df["path"]) for df in tables.values()]):
        try:
            durations[p] = sf.info(f"{BASE}/clips/{p}").duration
        except Exception:
            pass
    SUMMARY["source_durees"] = "en-têtes mp3"

for s, df in tables.items():
    df["duration"] = df["path"].map(durations)
    x = df["duration"].dropna()
    print(f"{s:12s} {x.sum() / 3600:8.1f} h | médiane {x.median():4.1f} s | min {x.min():4.1f} s | "
          f"max {x.max():5.1f} s | 99 % des clips < {x.quantile(0.99):4.1f} s | "
          f"> 15 s : {(x > 15).sum():,} | durée inconnue : {df['duration'].isna().sum():,}")
    SUMMARY[f"{s}_heures"] = round(float(x.sum() / 3600), 2)
    SUMMARY[f"{s}_duree_mediane_s"] = round(float(x.median()), 2)
    SUMMARY[f"{s}_duree_max_s"] = round(float(x.max()), 2)
    SUMMARY[f"{s}_duree_p99_s"] = round(float(x.quantile(0.99)), 2)

# Histogramme des durées : train officiel contre l'ensemble des clips validés
import matplotlib.pyplot as plt
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#52514e"
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.arange(0, 21, 0.5)
for s, color in [("validated", ORANGE), ("train", BLUE)]:
    if s in tables:
        ax.hist(tables[s]["duration"].dropna().clip(upper=20), bins=bins, color=color, alpha=0.7, label=s)
ax.set_title("Durée des clips (secondes)", loc="left")
ax.set_xlabel("Durée (s), les clips de plus de 20 s sont regroupés à droite")
ax.set_ylabel("Nombre de clips")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("/content/eda_durees.png", dpi=150)
plt.show()

## Cellule 5 — Point 2 : caractères et vocabulaire

On applique **notre normalisation actuelle** (`stt/text.py`) et on compte :
- le pourcentage de phrases **écartées** (un caractère hors de nos 49), et les heures perdues ;
- les **caractères hors vocabulaire** les plus fréquents (les caractères seuls, jamais les phrases).

**Décision :** si peu de phrases sont écartées (≈ 1,4 % sur le petit dataset), on garde le vocabulaire. Si un caractère fréquent manque, on discutera de l'ajouter.

In [ ]:
# Cellule 5 : point 2, caractères et vocabulaire
from collections import Counter
from stt.text import normalize, is_valid, ALLOWED

for s, df in tables.items():
    df["text"] = df["sentence"].map(normalize)                # notre normalisation actuelle
    df["valid"] = df["text"].map(is_valid)                    # True si tous les caractères sont dans nos 49
    rejected = ~df["valid"]
    lost_h = df.loc[rejected, "duration"].sum() / 3600
    print(f"{s:12s} phrases écartées : {rejected.sum():>7,} / {len(df):,} ({100 * rejected.mean():.2f} %) "
          f"soit {lost_h:.1f} h perdues")
    SUMMARY[f"{s}_phrases_ecartees_pct"] = round(float(100 * rejected.mean()), 2)

POOL = "validated" if "validated" in tables else "train"     # le plus grand tableau disponible
raw_chars = Counter("".join(tables[POOL]["sentence"]))
norm_chars = Counter("".join(tables[POOL]["text"]))
outside = sorted(((c, n) for c, n in norm_chars.items() if c not in ALLOWED), key=lambda x: -x[1])
print(f"\n{POOL} : {len(raw_chars)} caractères différents avant normalisation, {len(norm_chars)} après")
print(f"Caractères de notre vocabulaire jamais vus : {sorted(ALLOWED - set(norm_chars))}")
print(f"Phrases contenant un chiffre : {tables[POOL]['text'].str.contains(r'[0-9]').sum():,}")
print("Caractères hors vocabulaire les plus fréquents (après normalisation) :")
for c, n in outside[:30]:
    print(f"  {c!r:6} U+{ord(c):04X} : {n:,}")
SUMMARY["caracteres_avant_normalisation"] = len(raw_chars)
SUMMARY["caracteres_hors_vocabulaire"] = len(outside)
SUMMARY["top_hors_vocabulaire"] = {c: n for c, n in outside[:30]}

## Cellule 6 — Point 5 : contrainte du CTC

Le CTC exige au moins **un instant de sortie par caractère**, plus un blank entre deux lettres identiques. Après notre réduction ÷ 4, le modèle dispose de **25 instants par seconde**.

**À regarder :**
- le nombre de clips **impossibles** avec ÷ 4 (plus de 25 instants nécessaires par seconde). Il était de 0 sur le petit dataset ;
- les clips **suspects** (moins de 2 par seconde) : un audio trop long pour son texte, probablement du silence ou une mauvaise transcription.

In [ ]:
# Cellule 6 : point 5, contrainte du CTC (réduction ÷ 4 du nombre d'instants)
def ctc_needed(text):
    # instants minimum pour le CTC : un par caractère, plus un blank entre deux lettres identiques
    return len(text) + sum(a == b for a, b in zip(text, text[1:]))

for s in [s for s in ("train", "validated") if s in tables]:
    df = tables[s][tables[s]["valid"] & tables[s]["duration"].notna()]
    rate = df["text"].map(ctc_needed) / df["duration"]      # instants nécessaires par seconde
    print(f"{s:12s} médiane {rate.median():5.1f} | 99 % < {rate.quantile(0.99):5.1f} | max {rate.max():5.1f} "
          f"instants/s | impossibles avec ÷ 4 (> 25/s) : {(rate > 25).sum():,} | "
          f"suspects (< 2/s, audio trop long pour le texte) : {(rate < 2).sum():,}")
    SUMMARY[f"{s}_ctc_max_par_s"] = round(float(rate.max()), 2)
    SUMMARY[f"{s}_ctc_impossibles_div4"] = int((rate > 25).sum())
    SUMMARY[f"{s}_ctc_suspects"] = int((rate < 2).sum())

## Cellule 7 — Point 6 : locuteurs

Le nombre de locuteurs, la part du locuteur le plus actif et celle des 10 plus actifs, et les clips perdus si on plafonnait à 200 ou 100 clips par locuteur.

**Sur le petit dataset :** 275 locuteurs, et le top 10 représentait 44 à 54 % des clips. On espère beaucoup plus de voix, et un déséquilibre plus faible.

In [ ]:
# Cellule 7 : point 6, locuteurs
for s in [s for s in ("train", "dev", "test", "validated") if s in tables]:
    counts = tables[s]["client_id"].value_counts()           # nombre de clips par locuteur, du plus actif au moins actif
    total = counts.sum()
    print(f"{s:12s} {len(counts):>6,} locuteurs | médiane {counts.median():5.0f} clips | "
          f"le plus actif {counts.iloc[0]:,} ({100 * counts.iloc[0] / total:.1f} %) | "
          f"top 10 : {100 * counts.iloc[:10].sum() / total:.1f} % | "
          f"perdus si plafond 200 : {100 * (counts - 200).clip(lower=0).sum() / total:.1f} %, "
          f"plafond 100 : {100 * (counts - 100).clip(lower=0).sum() / total:.1f} %")
    SUMMARY[f"{s}_locuteurs"] = len(counts)
    SUMMARY[f"{s}_top10_pct"] = round(float(100 * counts.iloc[:10].sum() / total), 1)

## Cellule 8 — Point 7a : recouvrement à l'intérieur du grand dataset, et réserve utilisable

**Pourquoi :** si un locuteur ou une phrase du dev ou du test se trouve aussi dans le train, le WER sera **faussement bon** (c'est de la triche involontaire).

**La « réserve » :** les clips de `validated`, sans aucun locuteur ni aucune phrase du dev et du test. C'est le nombre d'heures **réellement utilisables** pour entraîner sans triche. Ce chiffre décidera entre le dataset entier et un sous-ensemble.

In [ ]:
# Cellule 8 : point 7a, recouvrement à l'intérieur du grand dataset
speakers = {s: set(df["client_id"]) for s, df in tables.items()}
sentences = {s: set(df["text"]) for s, df in tables.items()}
for s in ("dev", "test"):
    print(f"{s} : {len(speakers[s] & speakers['train']):,} locuteurs aussi dans le train | "
          f"{len(sentences[s] & sentences['train']):,} phrases aussi dans le train")
    SUMMARY[f"{s}_locuteurs_communs_train"] = len(speakers[s] & speakers["train"])

if "validated" in tables:
    # Réserve possible : clips validés, sans locuteur ni phrase du dev ou du test (pour éviter la triche)
    v = tables["validated"]
    held_speakers = speakers["dev"] | speakers["test"]
    held_sentences = sentences["dev"] | sentences["test"]
    usable = v[v["valid"] & ~v["client_id"].isin(held_speakers) & ~v["text"].isin(held_sentences)]
    in_train = usable["path"].isin(set(tables["train"]["path"])).sum()
    print(f"\nRéserve utilisable (validated sans dev/test) : {len(usable):,} clips | "
          f"{usable['duration'].sum() / 3600:.1f} h | {usable['client_id'].nunique():,} locuteurs | "
          f"dont {in_train:,} clips déjà dans train.tsv")
    SUMMARY["reserve_clips"] = len(usable)
    SUMMARY["reserve_heures"] = round(float(usable["duration"].sum() / 3600), 2)
    SUMMARY["reserve_locuteurs"] = int(usable["client_id"].nunique())

## Cellule 9 — Point 7b : recouvrement avec NOS données actuelles

On télécharge seulement les `.tsv` de notre petit dataset (fr-FR et fr-CA, quelques Mo), puis on regarde si nos clips et nos locuteurs se retrouvent dans le grand dataset.

**Pourquoi :** si nos clips de **dev** actuels sont dans le **grand train**, l'ancien dev ne pourra plus servir à comparer avec nos résultats précédents (72,5 % et 70,8 %). Il faudra alors définir de nouvelles règles d'évaluation.

In [ ]:
# Cellule 9 : point 7b, recouvrement avec NOS données actuelles (petit dataset)
import subprocess
SMALL = {"fr-FR": "cmrumhinj0001md07y3huuhs5", "fr-CA": "cmrumhr9h0005md07ph5dkxpp"}
small = {"train": [], "dev": [], "test": []}
for name, dataset_id in SMALL.items():
    tmp_dir = f"/tmp/small_{name}"
    download_dataset(dataset_id, download_directory=tmp_dir)
    archive = glob.glob(f"{tmp_dir}/*.tar.gz")[0]
    subprocess.run(["tar", "-xzf", archive, "-C", tmp_dir, "--wildcards", "*.tsv"], check=True)  # les .tsv seulement
    for s in small:
        path = glob.glob(f"{tmp_dir}/**/{s}.tsv", recursive=True)[0]
        small[s].append(pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE, low_memory=False))
    shutil.rmtree(tmp_dir)
small = {s: pd.concat(parts) for s, parts in small.items()}     # fr-FR + fr-CA réunis

big_train_paths = set(tables["train"]["path"])
big_pool_paths = set(usable["path"]) if "validated" in tables else big_train_paths
for s in ("train", "dev", "test"):
    paths, spk = set(small[s]["path"]), set(small[s]["client_id"])
    print(f"Notre {s:5s} ({len(paths):,} clips) : {len(paths & big_train_paths):,} clips dans le grand train | "
          f"{len(paths & big_pool_paths):,} dans la réserve | "
          f"locuteurs dans le grand train : {len(spk & speakers['train'])} / {len(spk)}")
    SUMMARY[f"petit_{s}_clips_dans_grand_train"] = len(paths & big_train_paths)
    SUMMARY[f"petit_{s}_locuteurs_dans_grand_train"] = len(spk & speakers["train"])

## Cellule 10 — Point 4 : fréquences d'échantillonnage et vitesse de lecture

Sur 300 clips du train tirés au hasard :
- les **fréquences d'échantillonnage** (32, 44,1 ou 48 kHz sur le petit dataset), toutes ramenées à 16 kHz ;
- le temps de **lecture + log-Mel** par clip. C'est exactement le travail du DataLoader. On en déduit le temps de préparation d'une époque avec 4 workers : s'il dépasse le temps de calcul du GPU, c'est le CPU qui ralentira l'entraînement.

In [ ]:
# Cellule 10 : point 4, fréquences d'échantillonnage et vitesse de lecture
import random, time
from stt.audio import load_audio, log_mel

clips_dir = f"{BASE}/clips"
if os.path.isdir(clips_dir):
    random.seed(0)
    sample = random.sample(list(tables["train"]["path"]), min(300, len(tables["train"])))
    rates = Counter(sf.info(f"{clips_dir}/{p}").samplerate for p in sample)
    print("Fréquences d'échantillonnage (300 clips du train) :", dict(rates))

    start = time.time()
    for p in sample:
        log_mel(load_audio(f"{clips_dir}/{p}"))              # exactement le travail du DataLoader
    per_clip = (time.time() - start) / len(sample)
    n_train = len(tables["train"])
    print(f"Lecture + log-Mel : {1000 * per_clip:.0f} ms par clip, soit pour une époque du train "
          f"({n_train:,} clips) ≈ {per_clip * n_train / 60:.0f} min sur 1 cœur, "
          f"≈ {per_clip * n_train / 4 / 60:.0f} min avec 4 workers")
    SUMMARY["frequences"] = {str(k): v for k, v in rates.items()}
    SUMMARY["ms_par_clip"] = round(1000 * per_clip, 1)
else:
    print("Clips non décompressés (place insuffisante) : point 4 impossible dans cette session")

## Cellule 11 — Sauvegarde des chiffres

Tous les chiffres sont enregistrés dans `/content/eda_grand_dataset.json`. Il ne contient aucune phrase : seulement des nombres, et les caractères hors vocabulaire. Il est ensuite téléchargé sur ton PC.

**C'est cette sortie qu'il faut me renvoyer.** On s'en servira pour remplir le journal et décider de la suite.

In [ ]:
# Cellule 11 : sauvegarde des chiffres (aucune phrase du dataset)
with open("/content/eda_grand_dataset.json", "w") as f:
    json.dump(SUMMARY, f, indent=1, ensure_ascii=False)
print(json.dumps(SUMMARY, indent=1, ensure_ascii=False))

from google.colab import files
files.download("/content/eda_grand_dataset.json")   # le résumé (aucune phrase) arrive dans tes Téléchargements